In [2]:
import pandas as pd

df = pd.read_parquet(
    "s3://dyrmgraph/invalid/silver/gkg/pub_date=20261022", storage_options={"key": "dg_mi_poc_user", "secret": "dg_mi_poc_pw", "client_kwargs": {"endpoint_url": "http://localhost:9000"}}
)

In [7]:
df.count()

GKGRecordID                     1245
V2_1Date                        1245
V2SourceCollectionIdentifier    1245
V2SourceCommonName              1245
V2DocumentIdentifier            1245
V1Counts                         149
V2_1Counts                       149
V1Themes                        1162
V2EnhancedThemes                1162
V1Locations                      966
V2EnhancedLocations              965
V1Persons                        882
V2EnhancedPersons                871
V1Organizations                  959
V2EnhancedOrganizations          931
V1_5Tone                        1245
V2_1EnhancedDates                506
V2GCAM                          1245
V2_1SharingImage                 987
V2_1RelatedImages                129
V2_1SocialImageEmbeds             37
V2_1SocialVideoEmbeds            634
V2_1Quotations                   218
V2_1AllNames                    1192
V2_1Amounts                      941
V2_1TranslationInfo                0
V2ExtraSXML                     1245
_

In [4]:
regex_rules = {
    "GKGRecordID": r"^\d{14}-T?\d+$",
    "V2_1Date": r"^(?:\d{14}|0)$",
    "V2SourceCollectionIdentifier": r"^\d+$",
    "V2SourceCommonName": r"^[^\t]*$",
    "V2DocumentIdentifier": r"^[^\t]*$",
    "V1Counts": r"^(?:([^#;]*#){9}[^#;]*(?:;([^#;]*#){9}[^#;]*)*)?$",
    "V2_1Counts": r"^(?:([^#;]*#){10}[^#;]*(?:;([^#;]*#){10}[^#;]*)*)?$",
    "V1Themes": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2EnhancedThemes": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*)?$",
    "V1Locations": r"^(?:([^#;]*#){6}[^#;]*(?:;([^#;]*#){6}[^#;]*)*)?$",
    "V2EnhancedLocations": r"^(?:([^#;]*#){8}[^#;]*(?:;([^#;]*#){8}[^#;]*)*)?$",
    "V1Persons": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2EnhancedPersons": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*)?$",
    "V1Organizations": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2EnhancedOrganizations": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*)?$",
    "V1_5Tone": r"^(?:([^,]*,){6}[^,]*)?$",
    "V2_1EnhancedDates": r"^(?:([^,;]*,){4}[^,;]*(?:;([^,;]*,){4}[^,;]*)*)?$",
    "V2GCAM": r"^(?:[^,:]+:[^,]*(?:,[^,:]+:[^,]*)*)?$",
    "V2_1SharingImage": r"^[^\t]*$",
    "V2_1RelatedImages": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2_1SocialImageEmbeds": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2_1SocialVideoEmbeds": r"^(?:[^;]*(?:;[^;]*)*)?$",
    "V2_1Quotations": r"^[^\t]*$",
    "V2_1AllNames": r"^(?:[^,;]*,[^,;]*(?:;[^,;]*,[^,;]*)*)?$",
    "V2_1Amounts": r"^(?:([^,;]*,){2}[^,;]*(?:;([^,;]*,){2}[^,;]*)*)?$",
    "V2_1TranslationInfo": r"^[^\t]*$",
    "V2ExtraSXML": r"^[^\t]*$",
}

In [8]:
invalid_results = []

for column, pattern in regex_rules.items():
    if column not in df.columns:
        print(f"Missing column: {column}")
        continue

    # Null values are trusted
    valid = df[column].astype("string").str.fullmatch(pattern).fillna(True)

    invalid = df.loc[~valid, [column]].copy()

    if not invalid.empty:
        invalid_results.append(pd.DataFrame({"row_index": invalid.index, "column": column, "invalid_value": invalid[column].to_numpy()}))

invalid_df = pd.concat(invalid_results, ignore_index=True) if invalid_results else pd.DataFrame(columns=["row_index", "column", "invalid_value"])

print(f"Total invalid values: {len(invalid_df)}")
display(invalid_df)

Total invalid values: 2909


,row_index,column,invalid_value
0,1,V1Counts,"ARREST#3##4#Delhi, Delhi, India#IN#IN07#28.666..."
1,10,V1Counts,PROTEST#200#movement x#1#France#FR#FR#46#2#FR;...
2,17,V1Counts,WOUND#8##1#Saudi Arabia#SA#SA#25#45#SA;CRISISL...
3,24,V1Counts,"KILL#100#urban#4#Rayachoti, Andhra Pradesh, In..."
4,25,V1Counts,KILL#7##1#Malaysia#MY#MY#2.5#112.5#MY;KILL#90#...
...,...,...,...
2904,1236,V2_1Amounts,"2,weeks before Election Day,1097;4000000,dolla..."
2905,1238,V2_1Amounts,"9,gigawatts of capacity not,1037;27000000000,b..."
2906,1240,V2_1Amounts,"1000000,of parallel execution environments,214..."
2907,1242,V2_1Amounts,"2000000,views,907;"


In [33]:
invalid_df[invalid_df["column"] == "V1Locations"]["invalid_value"][1460]

'4#Brussel, Bruxelles-Capitale, Belgium#BE#BE11#50.8333#4.33333#-1955538;1#Russia#RS#RS#60#100#RS;1#Georgia#GG#GG#41.999981#43.499905#GG;1#France#FR#FR#46#2#FR;1#China#CH#CH#35#105#CH;1#Finland#FI#FI#64#26#FI;4#Ankara, Ankara, Turkey#TU#TU68#39.9272#32.8644#-735338;4#Tbilisi, Georgia (General), Georgia#GG#GG00#41.725#44.7908#-2330077;1#Azerbaijan#AJ#AJ#40.5#47.5#AJ;1#Poland#PL#PL#52#20#PL;1#Ukraine#UP#UP#49#32#UP;1#Iran#IR#IR#32#53#IR;1#Armenia#AM#AM#40#45#AM;1#Georgia#GG#GG#42#43.5#GG;4#Baku, Baki*, Azerbaijan#AJ#AJ09#40.3953#49.8822#-2705195;1#Lithuania#LH#LH#56#24#LH;1#Germany#GM#GM#51.5#10.5#GM;1#Latvia#LG#LG#57#25#LG;4#Brussels, Bruxelles-Capitale, Belgium#BE#BE11#50.8333#4.33333#-1955538;1#Estonia#EN#EN#59#26#EN;0#Georgia, , Georgia#GG#GG#42#.5#1#GG'

In [9]:
summary = invalid_df.groupby("column").size().reset_index(name="invalid_count").sort_values("invalid_count", ascending=False)

display(summary)

,column,invalid_count
3,V2EnhancedThemes,1162
4,V2_1Amounts,941
6,V2_1EnhancedDates,506
0,V1Counts,149
5,V2_1Counts,149
2,V2EnhancedLocations,1
1,V1Locations,1
